# ICCIT2026 segpriors — Kaggle worker 4

Runs this account's slice of the channel-mode study (`ICCIT2026_MASTER_PLAN.md`) on **CVC-ClinicDB**
(ISIC18 runs on the SSH remote instead — see the note at the end of this cell).

**Assigned configs, in priority order (train x3 seeds [1337, 2024, 7] + eval each):**
- `experiment/iccit/unet_m5_clinicdb.yaml`
- `experiment/iccit/mkunet_m7_matched_clinicdb.yaml`
- `experiment/iccit/mkunet_m4_pre_clinicdb.yaml`
- `experiment/iccit/mkunet_m5_pre_clinicdb.yaml`

*Block D (U-Net, m5) + rest of Block B + all of Block C. Estimated ~6.25h.*

**Before running — Kaggle notebook settings:**
1. Settings -> Accelerator -> **GPU T4 x2** (or P100 — anything with CUDA works).
2. Settings -> Internet -> **On** (needed to clone the repo and install packages).
3. Add Data -> attach your `clinicdb-train-val-test-images-and-masks` dataset (or wherever `ClinicDB/train/images/...` lives) — the notebook searches `/kaggle/input/` for a `ClinicDB` directory, so the exact dataset name/owner doesn't matter.
4. Run all cells top to bottom. Sized for a **~9 hour** session (a 8.5h internal budget, see step 5, leaves margin for setup overhead).

Everything is pinned to commit `9115f5a` of the repo so every device in this study runs identical code.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
print(subprocess.run(["python3", "--version"], capture_output=True, text=True).stdout)


## 1. Clone the repo at the pinned commit

In [ ]:
%cd /kaggle/working
!rm -rf segpriors
!git clone https://github.com/Syfur007/segpriors.git segpriors
%cd segpriors
!git checkout 9115f5a
!git rev-parse HEAD


## 2. Reproduce the exact training environment (Python 3.8 + pinned deps)

Kaggle's default image is a newer Python than this repo's pinned stack (`requirements.txt` is
frozen against Python 3.8 — `pyarrow==17.0.0` is explicitly the last release with a 3.8 wheel).
Rather than fight version resolution against Kaggle's default interpreter, this builds a matching
Miniconda env once, so every device in the study (this notebook, the other 3, and the SSH remote)
runs the same interpreter + package versions — not just "close enough".

In [ ]:
import os
if not os.path.exists("/opt/conda_iccit"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-py38_23.11.0-2-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p /opt/conda_iccit
!/opt/conda_iccit/bin/conda --version


In [ ]:
PY = "/opt/conda_iccit/bin/python"
PIP = "/opt/conda_iccit/bin/pip"

!{PIP} install -q --upgrade pip
!{PIP} install -q torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!{PIP} install -q -r requirements.txt
!{PY} -c "import torch; print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())"


## 3. Attach the ClinicDB dataset

Kaggle mounts an attached dataset as an already-extracted, **read-only** directory tree under
`/kaggle/input/` — not a zip to unpack (e.g.
`/kaggle/input/datasets/<owner>/clinicdb-train-val-test-images-and-masks/ClinicDB/`). This searches
for a directory named `ClinicDB` containing `train/images` so the exact owner/slug doesn't matter,
then symlinks it into place (no copy needed — nothing in this pipeline writes back into the dataset
root).

In [ ]:
import glob, os

candidates = [
    p for p in glob.glob("/kaggle/input/**/ClinicDB", recursive=True)
    if os.path.isdir(os.path.join(p, "train", "images"))
]
assert candidates, (
    "No ClinicDB/ directory (containing train/images) found under /kaggle/input/. "
    "Add the dataset via the '+ Add Data' button (top right) before running this cell."
)
clinicdb_path = candidates[0]
print("Using:", clinicdb_path)

os.makedirs("data/polyp", exist_ok=True)
link_path = "data/polyp/ClinicDB"
if os.path.islink(link_path) or os.path.exists(link_path):
    os.remove(link_path) if os.path.islink(link_path) else None
os.symlink(clinicdb_path, link_path)

assert os.path.isdir("data/polyp/ClinicDB/train/images"), "Unexpected layout — check data/polyp/ClinicDB/"
print("OK — data/polyp/ClinicDB is ready.")
!ls data/polyp/ClinicDB


## 4. Sanity check (mirrors the pre-flight gate run on the other devices)

In [ ]:
import os
os.environ["PYTHONPATH"] = os.getcwd()
!{PY} -c "
from utils.config import load_config
from datasets import StandardSplitDataModule
cfg = load_config('configs/experiment/iccit/mkunet_m1_clinicdb.yaml')
dm = StandardSplitDataModule(cfg)
tl, vl = dm.get_standard_loaders()
print('train:', len(tl.dataset), 'val:', len(vl.dataset))
"


## 5. Run this notebook's assigned configs

Each config trains all 3 pre-registered seeds (`[1337, 2024, 7]`) via `orchestration.runner.run_sweep`
(writes `artifacts/runs/<run_id>/manifest.json` + ledger rows, exactly like the other devices), then
evaluates each seed's checkpoint. One config failing does not stop the rest — matches the "log it,
keep going" behaviour the rest of the pipeline already uses; check the printed status lines at the end.

Configs are listed **in priority order** (heaviest/most-important first). A 8.5h
time budget (leaving ~30min under the 9h session target for setup overhead) is checked before
starting each new config — the timing estimates behind this assignment are real measurements from
the other devices, but MK-UNet-T channel-mode configs are all close enough in cost that estimation
error is unlikely to matter; if it ever runs long anyway, this budget stops it from starting a config
it can't finish, rather than getting killed mid-run when the Kaggle session ends. Anything not
reached just doesn't run — nothing is left half-written.

In [ ]:
import time

CONFIGS = [
    "experiment/iccit/unet_m5_clinicdb.yaml",
    "experiment/iccit/mkunet_m7_matched_clinicdb.yaml",
    "experiment/iccit/mkunet_m4_pre_clinicdb.yaml",
    "experiment/iccit/mkunet_m5_pre_clinicdb.yaml"
]
BUDGET_SECONDS = 8.5 * 3600
start_time = time.time()

results = []
for cfg in CONFIGS:
    elapsed = time.time() - start_time
    remaining = BUDGET_SECONDS - elapsed
    print(f"\n[{elapsed/3600:.2f}h elapsed, {remaining/3600:.2f}h left in budget]")
    if remaining <= 0:
        print(f"Time budget exhausted — skipping {cfg} and everything after it.")
        results.append((cfg, "skipped (budget)", "skipped (budget)"))
        continue

    print(f"\n{'='*70}\n TRAIN {cfg}\n{'='*70}")
    rc_train = os.system(f"{PY} scripts/run_iccit_sweep.py --config configs/{cfg}")
    print(f"\n{'='*70}\n EVAL {cfg}\n{'='*70}")
    rc_eval = os.system(f"{PY} scripts/eval_iccit_sweep.py --config configs/{cfg}")
    results.append((cfg, rc_train, rc_eval))

print(f"\n\n=== SUMMARY (0 = ok, nonzero = at least one seed failed) — total {(time.time()-start_time)/3600:.2f}h ===")
for cfg, rc_t, rc_e in results:
    print(f"  {cfg:55s} train={rc_t} eval={rc_e}")


## 6. Package results for download

Kaggle keeps `/kaggle/working/` as this notebook version's Output after the session ends —
download `iccit_results_worker{idx}.zip` from the Output tab once this finishes.

In [ ]:
!cd /kaggle/working/segpriors && zip -qr /kaggle/working/iccit_results_worker4.zip artifacts/ checkpoints/ logs/ -x "*.pth"
!cd /kaggle/working/segpriors && zip -qr /kaggle/working/iccit_checkpoints_worker4.zip checkpoints/
!ls -lh /kaggle/working/*.zip
